## __Tentative de fine tuning de mobilenetV2__

In [1]:
import sys
import os
import pandas as pd

sys.path.append(os.path.abspath(".."))

from transformers import AutoImageProcessor, AutoModelForImageClassification
from torchmetrics.classification import MulticlassF1Score
from lib.data.preprocessing import TorchPreprocessor
from lib.data.dataset import BeeDataset
from lib.data.train_val_split import train_val_split
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import torch.optim as optim

import torch
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"L'entraînement se fera sur : {device}")

2026-03-02 13:53:22.642327: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


L'entraînement se fera sur : cuda


In [2]:
# Load model directly from Hugging Face
num_labels = 50
base_processor = AutoImageProcessor.from_pretrained("google/mobilenet_v2_1.0_224")
model = AutoModelForImageClassification.from_pretrained("google/mobilenet_v2_1.0_224", num_labels=num_labels,ignore_mismatched_sizes=True)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Some weights of MobileNetV2ForImageClassification were not initialized from the model checkpoint at google/mobilenet_v2_1.0_224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1001]) in the checkpoint and torch.Size([50]) in the model instantiated
- classifier.weight: found shape torch.Size([1001, 1280]) in the checkpoint and torch.Size([50, 1280]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Le preprocessor fourni ne pad pas, ce qui nous fait perdre des abeilles. On se propose donc de remplacer cette partie du code

In [3]:
mobile_net_processor_parameters= {
    "mean" : base_processor.image_mean,
    "std" : base_processor.image_std,
    "crop_size" : (base_processor.crop_size["height"], base_processor.crop_size["width"])
}

preprocessor = TorchPreprocessor(
    normalize=True,
    resize_method="pad",
    target_size=mobile_net_processor_parameters["crop_size"],
)

train_dataset, val_dataset = train_val_split(transform=preprocessor)

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [4]:
def evaluate(model, dataloader, device, metric_fn):
    model.eval()
    metric_fn.reset()
    total_loss = 0
    criterion = torch.nn.CrossEntropyLoss()

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            logits = outputs.logits
            
            # Calcul de la perte
            loss = criterion(logits, labels)
            total_loss += loss.item()
            
            # Calcul du F1
            preds = torch.argmax(logits, dim=-1)
            metric_fn.update(preds, labels)
            
    avg_loss = total_loss / len(dataloader)
    f1_score = metric_fn.compute()
    return avg_loss, f1_score

In [5]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

In [ ]:
# 1. Configuration (Backbone gelé, Tête active)
for param in model.parameters():
    param.requires_grad = False


for i in range(13, 16):
    for param in model.mobilenet_v2.layer[i].parameters():
        param.requires_grad = True

for param in model.mobilenet_v2.conv_1x1.parameters():
    param.requires_grad = True

for param in model.classifier.parameters():
    param.requires_grad = True

# Filtrer les paramètres pour l'optimiseur (plus propre pour la mémoire)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5)
criterion = torch.nn.CrossEntropyLoss()

# 2. Boucle d'entraînement
model.train()
num_epochs = 10

train_f1_metric = MulticlassF1Score(num_classes=num_labels, average='macro').to(device)
test_f1_metric = MulticlassF1Score(num_classes=num_labels, average='macro').to(device)
model.to(device)

for epoch in range(num_epochs):
    loop = tqdm(train_dataloader, desc=f"Epoch [{epoch+1}/{num_epochs}]", leave=True)
    
    epoch_loss = 0
    
    for images, labels in loop:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        logits = outputs.logits
        loss = criterion(logits, labels)
        
        loss.backward()
        optimizer.step()

        preds = torch.argmax(outputs.logits, dim=-1)
        train_f1_metric.update(preds, labels)
        
        epoch_loss += loss.item()
        loop.set_postfix(loss=f"{loss.item():.3f}", f1=f"{train_f1_metric.compute():.3f}")
        
    test_loss, test_f1 = evaluate(model, val_dataloader, device, test_f1_metric)

    # Affichage final de l'époque
    train_f1 = train_f1_metric.compute()
    print(f"-> Fin Epoch {epoch+1}")
    print(f"   TRAIN | Loss: {epoch_loss/len(train_dataloader):.4f} | F1: {train_f1:.4f}")
    print(f"   TEST  | Loss: {test_loss:.4f} | F1: {test_f1:.4f}")
    
    # Reset la métrique d'entraînement pour la prochaine époque
    train_f1_metric.reset()
    test_f1_metric.reset()

    # Score final de l'epoch
    print(f"-> Fin Epoch {epoch+1} | Loss: {epoch_loss/len(train_dataloader):.4f} | F1: {train_f1:.4f}")

MobileNetV2ForImageClassification(
  (mobilenet_v2): MobileNetV2Model(
    (conv_stem): MobileNetV2Stem(
      (first_conv): MobileNetV2ConvLayer(
        (convolution): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
        (normalization): BatchNorm2d(32, eps=0.001, momentum=0.997, affine=True, track_running_stats=True)
        (activation): ReLU6()
      )
      (conv_3x3): MobileNetV2ConvLayer(
        (convolution): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), groups=32, bias=False)
        (normalization): BatchNorm2d(32, eps=0.001, momentum=0.997, affine=True, track_running_stats=True)
        (activation): ReLU6()
      )
      (reduce_1x1): MobileNetV2ConvLayer(
        (convolution): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (normalization): BatchNorm2d(16, eps=0.001, momentum=0.997, affine=True, track_running_stats=True)
      )
    )
    (layer): ModuleList(
      (0): MobileNetV2InvertedResidual(
        (expand_1x1): MobileNe

Epoch [1/10]:   0%|          | 0/201 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
test_dataset = BeeDataset(train=False, transform=preprocessor)

model.eval()

all_results = [] 
with torch.no_grad():
    for images, ids in test_dataset:
        images = images.to(device)
        logits = model(images.unsqueeze(0))
        predicted_label = torch.argmax(logits, dim=-1).item() + 1  # +1 pour ajuster l'index si nécessaire
        
        all_results.append({"id": ids, "label": predicted_label})

submission_results = pd.DataFrame(all_results)

# Sauvegarde finale
submission_results.to_csv("submission1.csv", index=False)
print(f"✅ Terminé ! {len(submission_results)} prédictions sauvegardées.")

✅ Terminé ! 1997 prédictions sauvegardées.


In [ ]:
class JITWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        
    def forward(self, x):
        # On extrait uniquement le Tensor des logits pour satisfaire le JIT
        outputs = self.model(x)
        return outputs.logits

model.eval()
model.to("cpu") # L'export est plus stable sur CPU
wrapper = JITWrapper(model)

dummy_input = torch.randn(1, 3, 224, 224)
traced_model = torch.jit.trace(wrapper, dummy_input)

traced_model.save("models/mobilenet_v2_bees_jit.pt")

In [ ]:
loaded_jit_model = torch.jit.load("models/mobilenet_v2_bees_jit.pt")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loaded_jit_model.to(device)
loaded_jit_model.eval()

with torch.no_grad():
    # Assure-toi que les images sont aussi sur le même device
    images = images.to(device)
    results = loaded_jit_model(images)